In [8]:
import os
os.chdir("../..")

import copy
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.svm import LinearSVC
from spiking import (
    load_model,
    extract_features,
    extract_spike_times,
    evaluate_classifier,
    BinaryFirstSpike,
    LinearInversion,
    ScaledInversion,
    TargetRelative,
    NeuronMeanRelative,
)
from applications.datasets import create_dataset

ImportError: cannot import name 'extract_spike_times' from 'spiking' (/Users/Razvan/CodeUni/dummy-snn/spiking/__init__.py)

In [2]:
# --- Parameters ---
T_OBJECTIVE = 0.875
SEED = 1

model_dir = f"logs/mnist/threshold_research/tobj_{T_OBJECTIVE}/seed_{SEED}"

model = load_model(f"{model_dir}/model.pth")
layer = model.layers[-1]

with open(f"{model_dir}/setup.json") as f:
    setup = json.load(f)
t_objective = setup["t_objective"]

train_loader, test_loader = create_dataset("mnist")

print(f"Model: {model_dir}")
print(f"t_objective: {t_objective}")
print(f"Neurons: {layer.num_outputs}")
print(f"Training samples: {len(train_loader.dataset)}")

Model: logs/mnist/threshold_research/tobj_0.875/seed_1
t_objective: 0.875
Neurons: 256
Training samples: 60000


In [ ]:
def analyze_model(model, t_objective, train_loader, test_loader, decoder, title):
    """Run full spike time analysis and SVM evaluation on a model."""
    layer = model.layers[-1]
    num_outputs = layer.num_outputs
    preceding_layers = model.layers[:-1]
    total_samples = len(train_loader.dataset)
    batched_loader = DataLoader(train_loader.dataset, batch_size=256, shuffle=False)

    # --- Collect per-neuron spike times ---
    neuron_spike_times = [[] for _ in range(num_outputs)]
    with torch.no_grad():
        for batch_times, _labels in batched_loader:
            flat_input = batch_times.flatten(1)
            layer_input = flat_input
            for prev_layer in preceding_layers:
                layer_input = prev_layer.infer_spike_times_batch(layer_input)
            spike_times, _ = layer.infer_spike_times_and_potentials_batch(layer_input)
            st_np = spike_times.numpy()
            for n in range(num_outputs):
                finite_mask = np.isfinite(st_np[:, n])
                neuron_spike_times[n].append(st_np[finite_mask, n])
    neuron_spike_times = [np.concatenate(times) if times else np.array([]) for times in neuron_spike_times]

    spike_rates = np.array([len(times) / total_samples for times in neuron_spike_times])
    mean_spike_times = np.array([times.mean() if len(times) > 0 else np.nan for times in neuron_spike_times])

    print(f"--- {title} ---")
    print(f"Neurons that spike at least once: {np.sum(spike_rates > 0)} / {num_outputs}")
    print(f"Mean spike rate: {spike_rates.mean():.3f}")
    print(f"Mean spike time (across spiking neurons): {np.nanmean(mean_spike_times):.4f}")
    print()

    # --- Plot 1: Histogram of mean spike times ---
    spiking_means = mean_spike_times[~np.isnan(mean_spike_times)]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(spiking_means, bins=40, edgecolor="black", alpha=0.7)
    ax.axvline(t_objective, color="red", linestyle="--", linewidth=2, label=f"t_objective = {t_objective}")
    ax.set_xlabel("Mean spike time")
    ax.set_ylabel("Number of neurons")
    ax.set_title(f"{title}: Distribution of mean spike time per neuron")
    ax.legend()
    plt.tight_layout()
    plt.show()

    # --- Plot 2: Box plots for 20 most active neurons ---
    active_indices = np.argsort(spike_rates)[::-1][:20]
    fig, ax = plt.subplots(figsize=(12, 5))
    box_data = [neuron_spike_times[i] for i in active_indices]
    box_labels = [str(i) for i in active_indices]
    ax.boxplot(box_data, labels=box_labels, showfliers=False)
    ax.axhline(t_objective, color="red", linestyle="--", linewidth=2, label=f"t_objective = {t_objective}")
    ax.set_xlabel("Neuron index")
    ax.set_ylabel("Spike time")
    ax.set_title(f"{title}: Spike time distributions (20 most active neurons)")
    ax.legend()
    plt.tight_layout()
    plt.show()

    # --- Plot 3: Spike time heatmap ---
    n_bins = 50
    bin_edges = np.linspace(0, 1, n_bins + 1)
    sort_order = np.argsort(mean_spike_times)
    spiking_order = sort_order[~np.isnan(mean_spike_times[sort_order])]
    nonspiking_order = sort_order[np.isnan(mean_spike_times[sort_order])]
    sort_order = np.concatenate([spiking_order, nonspiking_order])

    heatmap = np.zeros((num_outputs, n_bins))
    for row, neuron_idx in enumerate(sort_order):
        times = neuron_spike_times[neuron_idx]
        if len(times) > 0:
            counts, _ = np.histogram(times, bins=bin_edges)
            heatmap[row] = counts

    fig, ax = plt.subplots(figsize=(12, 8))
    im = ax.imshow(heatmap, aspect="auto", cmap="hot", interpolation="nearest",
                   extent=[0, 1, num_outputs, 0])
    ax.axvline(t_objective, color="cyan", linestyle="--", linewidth=2, label=f"t_objective = {t_objective}")
    ax.set_xlabel("Spike time")
    ax.set_ylabel("Neuron (sorted by mean spike time)")
    ax.set_title(f"{title}: Spike time heatmap")
    ax.legend(loc="upper right")
    plt.colorbar(im, ax=ax, label="Count")
    plt.tight_layout()
    plt.show()

    # --- Plot 4: Spike rate vs mean spike time scatter ---
    fig, ax = plt.subplots(figsize=(8, 6))
    spiking_mask = spike_rates > 0
    ax.scatter(spike_rates[spiking_mask], mean_spike_times[spiking_mask], alpha=0.6, s=20)
    ax.axhline(t_objective, color="red", linestyle="--", linewidth=2, label=f"t_objective = {t_objective}")
    ax.set_xlabel("Spike rate (fraction of samples)")
    ax.set_ylabel("Mean spike time")
    ax.set_title(f"{title}: Spike rate vs mean spike time per neuron")
    ax.legend()
    plt.tight_layout()
    plt.show()

    # --- Plot 5: Winning spike time distribution ---
    all_first_spike_times = []
    with torch.no_grad():
        for batch_times, _labels in batched_loader:
            flat_input = batch_times.flatten(1)
            layer_input = flat_input
            for prev_layer in preceding_layers:
                layer_input = prev_layer.infer_spike_times_batch(layer_input)
            spike_times, _ = layer.infer_spike_times_and_potentials_batch(layer_input)
            first_times = spike_times.min(dim=1).values
            finite = torch.isfinite(first_times)
            all_first_spike_times.append(first_times[finite].numpy())
    all_first_spike_times = np.concatenate(all_first_spike_times)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(all_first_spike_times, bins=60, edgecolor="black", alpha=0.7)
    ax.axvline(t_objective, color="red", linestyle="--", linewidth=2, label=f"t_objective = {t_objective}")
    ax.set_xlabel("First spike time")
    ax.set_ylabel("Number of samples")
    ax.set_title(f"{title}: Distribution of winning (earliest) spike time per sample")
    ax.legend()
    plt.tight_layout()
    plt.show()

    n_no_spike = total_samples - len(all_first_spike_times)
    print(f"Samples with at least one spike: {len(all_first_spike_times)} / {total_samples} ({len(all_first_spike_times)/total_samples:.1%})")
    print(f"Mean first spike time: {all_first_spike_times.mean():.4f}")
    print(f"Std:  {all_first_spike_times.std():.4f}")
    print()

    # --- Plot 6a: Per-neuron mean spike time ± std (sorted decreasing) ---
    std_spike_times = np.array([times.std() if len(times) > 1 else 0.0 for times in neuron_spike_times])
    spiking_mask_all = ~np.isnan(mean_spike_times)
    sort_idx = np.argsort(mean_spike_times[spiking_mask_all])[::-1]
    sorted_means = mean_spike_times[spiking_mask_all][sort_idx]
    sorted_stds = std_spike_times[spiking_mask_all][sort_idx]

    x = np.arange(len(sorted_means))
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(x, sorted_means, width=1.0, edgecolor="none", alpha=0.7)
    ax.fill_between(x, sorted_means - sorted_stds, sorted_means + sorted_stds, alpha=0.25, color="C0")
    ax.axhline(t_objective, color="red", linestyle="--", linewidth=2, label=f"t_objective = {t_objective}")
    ax.set_xlabel("Neuron rank")
    ax.set_ylabel("Mean spike time")
    ax.set_title(f"{title}: Per-neuron mean spike time ± std (sorted decreasing)")
    ax.legend()
    plt.tight_layout()
    plt.show()

    # --- Plot 6b: Per-neuron mean winning spike time ± std (sorted decreasing) ---
    winner_spike_times = [[] for _ in range(num_outputs)]
    with torch.no_grad():
        for batch_times, _labels in batched_loader:
            flat_input = batch_times.flatten(1)
            layer_input = flat_input
            for prev_layer in preceding_layers:
                layer_input = prev_layer.infer_spike_times_batch(layer_input)
            spike_times, _ = layer.infer_spike_times_and_potentials_batch(layer_input)
            min_times, winners = spike_times.min(dim=1)
            finite = torch.isfinite(min_times)
            for t, w in zip(min_times[finite].numpy(), winners[finite].numpy()):
                winner_spike_times[w].append(t)

    win_counts = np.array([len(times) for times in winner_spike_times])
    win_means = np.array([np.mean(times) if len(times) > 0 else np.nan for times in winner_spike_times])
    win_stds = np.array([np.std(times) if len(times) > 1 else 0.0 for times in winner_spike_times])

    spiking_mask_win = ~np.isnan(win_means)
    sorted_indices = np.argsort(win_means[spiking_mask_win])[::-1]
    sorted_win_means = win_means[spiking_mask_win][sorted_indices]
    sorted_win_stds = win_stds[spiking_mask_win][sorted_indices]

    x = np.arange(len(sorted_win_means))
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(x, sorted_win_means, width=1.0, edgecolor="none", alpha=0.7)
    ax.fill_between(x, sorted_win_means - sorted_win_stds, sorted_win_means + sorted_win_stds, alpha=0.25, color="C0")
    ax.axhline(t_objective, color="red", linestyle="--", linewidth=2, label=f"t_objective = {t_objective}")
    ax.set_xlabel("Neuron rank")
    ax.set_ylabel("Mean winning spike time")
    ax.set_title(f"{title}: Per-neuron mean winning spike time ± std (sorted decreasing)")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"Neurons that win at least once: {np.sum(win_counts > 0)} / {num_outputs}")
    print(f"Most wins: neuron {win_counts.argmax()} with {win_counts.max()} wins")
    print()

    # --- Plot 7: SVM evaluation using decoder ---
    spike_times_train, y_train = extract_spike_times(model, train_loader, shape=(1, 28, 28))
    spike_times_test, y_test = extract_spike_times(model, test_loader, shape=(1, 28, 28))

    X_train = decoder.decode(spike_times_train).numpy()
    X_test = decoder.decode(spike_times_test).numpy()

    svm = LinearSVC(dual=False, tol=1e-3, max_iter=10000)
    svm.fit(X_train, y_train.numpy())

    train_acc = svm.score(X_train, y_train.numpy())
    test_acc = svm.score(X_test, y_test.numpy())
    print(f"SVM accuracy — train: {train_acc:.4f}, test: {test_acc:.4f}")

    feature_importance = np.abs(svm.coef_).mean(axis=0)
    drift = mean_spike_times - t_objective
    abs_drift = np.abs(drift)
    mask = ~np.isnan(mean_spike_times)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    ax.scatter(drift[mask], feature_importance[mask], alpha=0.5, s=20)
    ax.set_xlabel("Drift from t_objective (signed)")
    ax.set_ylabel("SVM feature importance")
    ax.set_title(f"{title}: Signed drift vs feature importance")
    r_signed = np.corrcoef(drift[mask], feature_importance[mask])[0, 1]
    ax.annotate(f"r = {r_signed:.3f}", xy=(0.05, 0.95), xycoords="axes fraction", fontsize=12, va="top")

    ax = axes[1]
    ax.scatter(abs_drift[mask], feature_importance[mask], alpha=0.5, s=20)
    ax.set_xlabel("|Drift| from t_objective")
    ax.set_ylabel("SVM feature importance")
    ax.set_title(f"{title}: Absolute drift vs feature importance")
    r_abs = np.corrcoef(abs_drift[mask], feature_importance[mask])[0, 1]
    ax.annotate(f"r = {r_abs:.3f}", xy=(0.05, 0.95), xycoords="axes fraction", fontsize=12, va="top")

    plt.tight_layout()
    plt.show()

    # --- Summary statistics ---
    n_spiking = np.sum(spike_rates > 0)
    n_silent = num_outputs - n_spiking

    print()
    print(f"=== Summary: {title} (t_objective = {t_objective}) ===")
    print(f"Total neurons: {num_outputs}")
    print(f"Spiking neurons: {n_spiking} ({n_spiking / num_outputs:.1%})")
    print(f"Silent neurons: {n_silent} ({n_silent / num_outputs:.1%})")
    print()

    if n_spiking > 0:
        all_finite_times = np.concatenate([t for t in neuron_spike_times if len(t) > 0])
        print(f"Spike times (all spikes):")
        print(f"  Mean:   {all_finite_times.mean():.4f}")
        print(f"  Std:    {all_finite_times.std():.4f}")
        print(f"  Min:    {all_finite_times.min():.4f}")
        print(f"  Max:    {all_finite_times.max():.4f}")
        print(f"  Median: {np.median(all_finite_times):.4f}")
        print()
        print(f"Gap from t_objective:")
        print(f"  Mean gap (per neuron): {np.nanmean(np.abs(mean_spike_times - t_objective)):.4f}")
        print(f"  Mean signed gap:       {np.nanmean(mean_spike_times - t_objective):.4f}")
        print()
        print(f"Spike rates:")
        print(f"  Mean: {spike_rates[spike_rates > 0].mean():.3f}")
        print(f"  Std:  {spike_rates[spike_rates > 0].std():.3f}")
        print(f"  Min:  {spike_rates[spike_rates > 0].min():.3f}")
        print(f"  Max:  {spike_rates[spike_rates > 0].max():.3f}")

In [ ]:
decoder = TargetRelative(t_target=t_objective)
analyze_model(model, t_objective, train_loader, test_loader, decoder=decoder, title="Original thresholds")

In [ ]:
with open(f"{model_dir}/optimal_thresholds.json") as f:
    opt_data = json.load(f)

model_optimal = copy.deepcopy(model)
model_optimal.layers[-1].thresholds = torch.nn.Parameter(torch.tensor(opt_data["optimal_thresholds"]))

decoder = TargetRelative(t_target=t_objective)
analyze_model(model_optimal, t_objective, train_loader, test_loader, decoder=decoder, title="Optimal thresholds")